# Module 8 — Hyperparameter Tuning: GridSearch vs. Random vs. Bayesian

> **Track 1 · Data Science Foundations**  
> **Difficulty**: Intermediate → Advanced  
> **Runtime**: ~8 minutes (tuning takes time)  
> **Key Libraries**: Scikit-learn, XGBoost, Optuna, NumPy, Pandas, Plotly, Seaborn  
> **Prerequisite**: Module 7 (Imbalanced Data)

## Table of Contents

1. [Business Context](#1-business-context)
2. [Setup & Imports](#2-setup--imports)
3. [Synthetic Credit Scoring Dataset](#3-synthetic-credit-scoring-dataset)
4. [Baseline Model](#4-baseline-model)
5. [Grid Search (Exhaustive)](#5-grid-search-exhaustive)
6. [Random Search (Stochastic)](#6-random-search-stochastic)
7. [Bayesian Optimization with Optuna](#7-bayesian-optimization-with-optuna)
8. [Comparison Dashboard](#8-comparison-dashboard)
9. [Key Business Takeaway](#9-key-business-takeaway)

---
## §1 · Business Context

### Why does hyperparameter tuning matter?

At **Revolut**, the credit-risk team uses XGBoost to predict loan defaults.
The model has **20+ hyperparameters**, and the default settings leave **10–15% AUC on the table**.

| Tuning Strategy | Time Cost | Quality | When to Use |
|-----------------|-----------|---------|-------------|
| **Grid Search** | High (exponential) | Good for 2–3 params | Small search space, final polish |
| **Random Search** | Medium | Good for 5–10 params | Broad exploration, quick wins |
| **Bayesian (Optuna)** | Low–Medium | Best for 10+ params | Production tuning, complex models |

**The business impact**: a 1% AUC improvement in credit scoring can reduce defaults by **£2M/year**
on a £200M loan book.

In this notebook we will:
1. Generate a synthetic credit-scoring dataset (50K applicants).
2. Tune XGBoost using all three strategies.
3. Compare wall-clock time vs. achieved AUC.
4. Visualize the search trajectory and parameter importance.

---
## §2 · Setup & Imports

In [ ]:
# ── Reproducibility ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import time
import warnings

np.random.seed(42)
warnings.filterwarnings("ignore")

# ── Scikit-learn ─────────────────────────────────────────────────
from sklearn.model_selection import (
    train_test_split, GridSearchCV, RandomizedSearchCV, StratifiedKFold
)
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report

# ── XGBoost ──────────────────────────────────────────────────────
from xgboost import XGBClassifier

# ── Optuna (Bayesian optimization) ───────────────────────────────
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Visualization ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# ── Display settings ─────────────────────────────────────────────
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
sns.set_theme(style="whitegrid")

print("✓ All imports loaded")
print(f"  Optuna {optuna.__version__}")

---
## §3 · Synthetic Credit Scoring Dataset

We simulate **50,000 loan applications** with 15 features and a ~15% default rate.

In [ ]:
# ── Configuration ────────────────────────────────────────────────
N_APPLICANTS = 50_000
DEFAULT_RATE = 0.15

print(f"Generating credit-scoring dataset:")
print(f"  Applicants : {N_APPLICANTS:,}")
print(f"  Default rate: {DEFAULT_RATE*100:.0f}%")

# ── Features ─────────────────────────────────────────────────────
np.random.seed(42)

data = {
    "age":                    np.random.randint(18, 70, N_APPLICANTS),
    "income":                 np.random.lognormal(10.5, 0.6, N_APPLICANTS),
    "employment_years":       np.random.exponential(5, N_APPLICANTS).clip(0, 30),
    "credit_score":           np.random.normal(650, 80, N_APPLICANTS).clip(300, 850),
    "loan_amount":            np.random.lognormal(9.5, 0.8, N_APPLICANTS),
    "loan_to_income":         np.random.uniform(0.1, 0.8, N_APPLICANTS),
    "num_credit_lines":       np.random.poisson(4, N_APPLICANTS),
    "num_late_payments":      np.random.poisson(0.5, N_APPLICANTS),
    "debt_to_income":         np.random.beta(2, 5, N_APPLICANTS),
    "home_ownership_rent":    np.random.binomial(1, 0.35, N_APPLICANTS),
    "purpose_debt_consolidation": np.random.binomial(1, 0.40, N_APPLICANTS),
    "interest_rate":          np.random.uniform(5, 25, N_APPLICANTS),
    "loan_term_months":       np.random.choice([12, 24, 36, 48, 60], N_APPLICANTS),
    "recent_credit_inquiries":np.random.poisson(1.5, N_APPLICANTS),
    "delinquency_2yrs":       np.random.poisson(0.3, N_APPLICANTS),
}

X = pd.DataFrame(data)

# ── Target: Default probability (logistic function) ──────────────
# Simulate a realistic default mechanism
log_odds = (
    -3.0
    - 0.02 * (X["credit_score"] - 650)
    + 0.03 * X["debt_to_income"] * 10
    + 0.5 * X["num_late_payments"]
    + 0.3 * X["delinquency_2yrs"]
    - 0.01 * X["income"] / 10000
    + 0.1 * X["interest_rate"]
    + np.random.normal(0, 1, N_APPLICANTS)  # noise
)
prob_default = 1 / (1 + np.exp(-log_odds))
y = (prob_default > 0.5).astype(int)

# Adjust to target default rate
threshold = np.percentile(prob_default, (1 - DEFAULT_RATE) * 100)
y = (prob_default > threshold).astype(int)

print(f"\n✓ Shape: {X.shape}")
print(f"  Default rate: {y.mean() * 100:.1f}%")
print(f"  Features: {X.shape[1]}")

X.head()

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

print(f"Train: {len(y_train):,} samples ({y_train.sum():,} defaults)")
print(f"Test : {len(y_test):,} samples ({y_test.sum():,} defaults)")

---
## §4 · Baseline Model (Default Hyperparameters)

In [ ]:
# Train XGBoost with default settings
xgb_baseline = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.3,
    random_state=42,
    eval_metric="logloss",
    use_label_encoder=False,
)

t0 = time.time()
xgb_baseline.fit(X_train, y_train)
train_time_baseline = time.time() - t0

y_pred_baseline = xgb_baseline.predict(X_test)
y_prob_baseline = xgb_baseline.predict_proba(X_test)[:, 1]

auc_baseline = roc_auc_score(y_test, y_prob_baseline)
acc_baseline = accuracy_score(y_test, y_pred_baseline)

print("=" * 60)
print("BASELINE XGBoost (Default Hyperparameters)")
print("=" * 60)
print(f"Train time    : {train_time_baseline:.2f}s")
print(f"ROC-AUC       : {auc_baseline:.4f}")
print(f"Accuracy      : {acc_baseline:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_baseline))

print("💡 This is our starting point — let's see how much tuning can improve it!")

---
## §5 · Grid Search (Exhaustive)

Grid Search evaluates **every combination** in the parameter grid.
For 3 parameters with 3 values each → 3³ = 27 combinations × 5 CV folds = 135 model fits.

In [ ]:
# Define parameter grid (small for speed)
param_grid = {
    "max_depth":       [3, 5, 7],
    "learning_rate":   [0.01, 0.1, 0.3],
    "n_estimators":    [50, 100, 200],
    "subsample":       [0.8, 1.0],
}

total_combinations = np.prod([len(v) for v in param_grid.values()])
print(f"Grid Search Configuration:")
print(f"  Parameters   : {len(param_grid)}")
print(f"  Combinations : {total_combinations}")
print(f"  CV folds     : 5")
print(f"  Total fits   : {total_combinations * 5}")

# Run Grid Search
grid_search = GridSearchCV(
    XGBClassifier(random_state=42, eval_metric="logloss", use_label_encoder=False),
    param_grid=param_grid,
    scoring="roc_auc",
    cv=5,
    n_jobs=-1,
    verbose=0,
)

print("\nRunning Grid Search …")
t0 = time.time()
grid_search.fit(X_train, y_train)
time_grid = time.time() - t0

print(f"✓ Completed in {time_grid:.1f}s")
print(f"\nBest parameters:")
for param, value in grid_search.best_params_.items():
    print(f"  {param:20s}: {value}")

print(f"\nBest CV AUC   : {grid_search.best_score_:.4f}")
print(f"Test AUC      : {roc_auc_score(y_test, grid_search.best_estimator_.predict_proba(X_test)[:, 1]):.4f}")

# Results
grid_results = pd.DataFrame(grid_search.cv_results_)
print(f"\nTop 5 configurations:")
print(grid_results.sort_values("rank_test_score").head(5)[
    ["params", "mean_test_score", "std_test_score", "mean_fit_time"]
].to_string(index=False))

---
## §6 · Random Search (Stochastic)

Random Search samples **random combinations** from the parameter space.
More efficient than grid search when many parameters have little effect.

In [ ]:
# Define parameter distributions
param_distributions = {
    "max_depth":       [3, 4, 5, 6, 7, 8, 10],
    "learning_rate":   [0.01, 0.05, 0.1, 0.2, 0.3],
    "n_estimators":    [50, 100, 150, 200, 300, 500],
    "subsample":       [0.6, 0.7, 0.8, 0.9, 1.0],
    "colsample_bytree":[0.6, 0.7, 0.8, 0.9, 1.0],
    "min_child_weight":[1, 3, 5, 7, 10],
    "gamma":           [0, 0.1, 0.2, 0.5],
}

n_iterations = 50  # sample 50 random combinations

print(f"Random Search Configuration:")
print(f"  Parameters   : {len(param_distributions)}")
print(f"  Iterations   : {n_iterations}")
print(f"  CV folds     : 5")
print(f"  Total fits   : {n_iterations * 5}")

# Run Random Search
random_search = RandomizedSearchCV(
    XGBClassifier(random_state=42, eval_metric="logloss", use_label_encoder=False),
    param_distributions=param_distributions,
    n_iter=n_iterations,
    scoring="roc_auc",
    cv=5,
    n_jobs=-1,
    random_state=42,
    verbose=0,
)

print("\nRunning Random Search …")
t0 = time.time()
random_search.fit(X_train, y_train)
time_random = time.time() - t0

print(f"✓ Completed in {time_random:.1f}s")
print(f"\nBest parameters:")
for param, value in random_search.best_params_.items():
    print(f"  {param:20s}: {value}")

print(f"\nBest CV AUC   : {random_search.best_score_:.4f}")
print(f"Test AUC      : {roc_auc_score(y_test, random_search.best_estimator_.predict_proba(X_test)[:, 1]):.4f}")

# Results
random_results = pd.DataFrame(random_search.cv_results_)
print(f"\nTop 5 configurations:")
print(random_results.sort_values("rank_test_score").head(5)[
    ["params", "mean_test_score", "std_test_score", "mean_fit_time"]
].to_string(index=False))

---
## §7 · Bayesian Optimization with Optuna

Bayesian optimization builds a **probabilistic model** of the objective function
and uses it to select the most promising hyperparameters to evaluate next.

In [ ]:
# Define Optuna objective function
def objective(trial):
    params = {
        "max_depth":        trial.suggest_int("max_depth", 3, 10),
        "learning_rate":    trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "n_estimators":     trial.suggest_int("n_estimators", 50, 500),
        "subsample":        trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma":            trial.suggest_float("gamma", 0, 0.5),
        "reg_alpha":        trial.suggest_float("reg_alpha", 0, 10),
        "reg_lambda":       trial.suggest_float("reg_lambda", 0, 10),
    }
    
    model = XGBClassifier(
        **params,
        random_state=42,
        eval_metric="logloss",
        use_label_encoder=False,
    )
    
    # Cross-validation
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    auc_scores = []
    
    for train_idx, val_idx in cv.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train[train_idx], y_train[val_idx]
        
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
        y_prob = model.predict_proba(X_val)[:, 1]
        auc_scores.append(roc_auc_score(y_val, y_prob))
    
    return np.mean(auc_scores)

# Run Optuna study
n_trials = 50

print(f"Bayesian Optimization (Optuna):")
print(f"  Trials       : {n_trials}")
print(f"  CV folds     : 5")
print(f"  Parameters   : 9")
print(f"\nRunning optimization …")

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))

t0 = time.time()
study.optimize(objective, n_trials=n_trials)
time_optuna = time.time() - t0

print(f"✓ Completed in {time_optuna:.1f}s")
print(f"\nBest trial: #{study.best_trial.number}")
print(f"Best CV AUC: {study.best_value:.4f}")
print(f"\nBest parameters:")
for param, value in study.best_params.items():
    print(f"  {param:20s}: {value}")

# Evaluate on test set
best_model = XGBClassifier(
    **study.best_params,
    random_state=42,
    eval_metric="logloss",
    use_label_encoder=False,
)
best_model.fit(X_train, y_train)
y_prob_optuna = best_model.predict_proba(X_test)[:, 1]
auc_optuna = roc_auc_score(y_test, y_prob_optuna)

print(f"\nTest AUC: {auc_optuna:.4f}")

---
## §8 · Comparison Dashboard

In [ ]:
# ── Compile results ──────────────────────────────────────────────
comparison = pd.DataFrame({
    "Method": ["Baseline", "Grid Search", "Random Search", "Bayesian (Optuna)"],
    "Time (s)": [train_time_baseline, time_grid, time_random, time_optuna],
    "Best CV AUC": [auc_baseline, grid_search.best_score_, random_search.best_score_, study.best_value],
    "Test AUC": [
        auc_baseline,
        roc_auc_score(y_test, grid_search.best_estimator_.predict_proba(X_test)[:, 1]),
        roc_auc_score(y_test, random_search.best_estimator_.predict_proba(X_test)[:, 1]),
        auc_optuna,
    ],
    "Total Fits": [1, total_combinations * 5, n_iterations * 5, n_trials * 5],
}).round(4)

comparison["AUC Improvement"] = ((comparison["Test AUC"] - comparison["Test AUC"].iloc[0]) / 
                                   comparison["Test AUC"].iloc[0] * 100).round(2)

print("=" * 90)
print("HYPERPARAMETER TUNING COMPARISON")
print("=" * 90)
print(comparison.to_string(index=False))

In [ ]:
# ── 8.1 Time vs. AUC Trade-off ───────────────────────────────────
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=comparison["Time (s)"],
    y=comparison["Test AUC"],
    mode="markers+text",
    text=comparison["Method"],
    textposition="top center",
    marker=dict(
        size=comparison["Total Fits"] / 10,
        color=["#636EFA", "#EF553B", "#00CC96", "#FFA15A"],
        line=dict(width=2, color="white"),
    ),
))

fig.update_layout(
    title="Time vs. Test AUC (bubble size = total model fits)",
    xaxis_title="Wall-Clock Time (seconds)",
    yaxis_title="Test ROC-AUC",
    height=450,
    xaxis_type="log",
)
fig.show()

print("💡 Bayesian optimization achieves the best AUC with fewer trials")
print("   Grid search is exhaustive but slow; random search is a good middle ground")

In [ ]:
# ── 8.2 Optuna Optimization History ──────────────────────────────
fig = go.Figure()

trial_numbers = [t.number for t in study.trials]
trial_values = [t.value for t in study.trials]
best_so_far = np.maximum.accumulate(trial_values)

fig.add_trace(go.Scatter(
    x=trial_numbers, y=trial_values,
    mode="markers",
    name="Trial AUC",
    marker=dict(size=8, color="#636EFA", opacity=0.6),
))

fig.add_trace(go.Scatter(
    x=trial_numbers, y=best_so_far,
    mode="lines",
    name="Best So Far",
    line=dict(color="#EF553B", width=3),
))

fig.update_layout(
    title="Optuna Optimization History",
    xaxis_title="Trial Number",
    yaxis_title="Cross-Validation AUC",
    height=420,
    legend=dict(orientation="h", y=1.12),
)
fig.show()

print("💡 Bayesian optimization quickly converges to good solutions")
print("   (red line shows the best AUC found so far)")

In [ ]:
# ── 8.3 Parameter Importance (Optuna) ────────────────────────────
importance = optuna.importance.get_param_importances(study)
importance_df = pd.DataFrame({
    "Parameter": list(importance.keys()),
    "Importance": list(importance.values()),
}).sort_values("Importance", ascending=True)

fig = px.bar(
    importance_df,
    x="Importance",
    y="Parameter",
    orientation="h",
    title="Hyperparameter Importance (Optuna)",
    color="Importance",
    color_continuous_scale="Viridis",
)
fig.update_layout(height=400, yaxis_title="")
fig.show()

print("💡 Focus tuning effort on the most important parameters")
print("   (learning_rate and max_depth typically dominate XGBoost)")

In [ ]:
# ── 8.4 Best Parameters Comparison ───────────────────────────────
best_params_df = pd.DataFrame({
    "Parameter": ["max_depth", "learning_rate", "n_estimators", "subsample",
                  "colsample_bytree", "min_child_weight", "gamma"],
    "Grid Search": [
        grid_search.best_params_.get("max_depth", "N/A"),
        grid_search.best_params_.get("learning_rate", "N/A"),
        grid_search.best_params_.get("n_estimators", "N/A"),
        grid_search.best_params_.get("subsample", "N/A"),
        "N/A", "N/A", "N/A",
    ],
    "Random Search": [
        random_search.best_params_.get("max_depth", "N/A"),
        random_search.best_params_.get("learning_rate", "N/A"),
        random_search.best_params_.get("n_estimators", "N/A"),
        random_search.best_params_.get("subsample", "N/A"),
        random_search.best_params_.get("colsample_bytree", "N/A"),
        random_search.best_params_.get("min_child_weight", "N/A"),
        random_search.best_params_.get("gamma", "N/A"),
    ],
    "Bayesian (Optuna)": [
        study.best_params.get("max_depth", "N/A"),
        round(study.best_params.get("learning_rate", 0), 4),
        study.best_params.get("n_estimators", "N/A"),
        round(study.best_params.get("subsample", 0), 3),
        round(study.best_params.get("colsample_bytree", 0), 3),
        study.best_params.get("min_child_weight", "N/A"),
        round(study.best_params.get("gamma", 0), 3),
    ],
})

print("=" * 80)
print("BEST HYPERPARAMETERS BY METHOD")
print("=" * 80)
print(best_params_df.to_string(index=False))

---
## §9 · Key Business Takeaway

> ### 🎯 Action Items for a Data Scientist
>
> | Method | Best For | Time Budget | Expected Improvement |
> |--------|----------|-------------|----------------------|
> | **Grid Search** | 2–3 parameters, final polish | High (hours) | +2–5% AUC |
> | **Random Search** | 5–10 parameters, exploration | Medium (30 min) | +3–7% AUC |
> | **Bayesian (Optuna)** | 10+ parameters, production | Low–Medium (1 hr) | +5–10% AUC |
>
> ### 💡 Revolut-Specific Guidelines
>
> 1. **Start with random search** for broad exploration (50–100 iterations).
>    - Covers more parameter space than grid search.
>    - Identifies which parameters matter most.
>
> 2. **Use Optuna for production models** where every 0.1% AUC counts.
>    - Bayesian optimization is sample-efficient (fewer trials needed).
>    - Automatically balances exploration vs. exploitation.
>
> 3. **Always use cross-validation** — never tune on the test set:
>    ```python
>    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
>    ```
>
> 4. **Log all experiments** with MLflow or Weights & Biases:
>    - Track which hyperparameters were tried.
>    - Reproduce the best model later.
>
> 5. **Parameter importance** guides future tuning:
>    - Focus on `learning_rate`, `max_depth`, `n_estimators` for XGBoost.
>    - Ignore parameters with < 5% importance.
>
> ### 📌 XGBoost Tuning Cheat Sheet
>
> ```python
> # Priority 1: Learning rate and trees
> "learning_rate":   0.01–0.3     (lower = better, needs more trees)
> "max_depth":       3–10         (higher = more complex, risk of overfit)
> "n_estimators":    50–1000      (more = better, but slower)
>
> # Priority 2: Regularization
> "subsample":       0.6–1.0      (row sampling, prevents overfit)
> "colsample_bytree": 0.6–1.0     (column sampling, prevents overfit)
> "min_child_weight": 1–10        (higher = more conservative)
> "gamma":           0–5          (min loss reduction for split)
> "reg_alpha":       0–10         (L1 regularization)
> "reg_lambda":      0–10         (L2 regularization)
> ```
>
> ### 🔑 Key Takeaways
>
> 1. **Default hyperparameters leave 5–15% AUC on the table** — always tune.
> 2. **Random search > Grid search** for high-dimensional spaces.
> 3. **Bayesian optimization** is the gold standard for production models.
> 4. **Cross-validation** prevents overfitting to the validation set.
> 5. **Parameter importance** tells you where to focus tuning effort.